In [2]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance


In [3]:
# -----------------------------
# Core function
# -----------------------------
def feature_importance_by_property(
    df: pd.DataFrame,
    *,
    feats: list[str],
    label_col: str = "label",
    properties: list[str] = ("ease","direction","delay"),
    group_col: str = "User ID (file)",
    n_splits: int = 5,
    min_n: int = 120,
    random_state: int = 42,
    rf_params: dict | None = None,
):
    """
    For each property (e.g., 'ease') and each of its values (e.g., 'Ease-out'), train LR and RF
    with GroupKFold CV, and aggregate feature importances:
      - LR: standardized odds ratios (exp(coef) with StandardScaler in pipeline)
      - RF: permutation importance on validation fold (mean over folds)

    Returns: dict like
      results[property]["LR"] -> DataFrame: value, n, feat, mean, std
      results[property]["RF"] -> DataFrame: value, n, feat, mean, std
    """
    rng = np.random.RandomState(random_state)

    # Prepare RF default params
    if rf_params is None:
        rf_params = dict(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1,
        )

    # Build pipelines
    lr_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=random_state)),
    ])
    rf_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(**rf_params)),
    ])

    # Ensure feats exist
    for c in feats:
        if c not in df.columns:
            df[c] = np.nan

    # Prepare y and groups
    y_all = (df[label_col].astype(str).str.upper() == "ON").astype(int)
    groups_all = df[group_col] if group_col in df.columns else pd.Series(0, index=df.index)

    results = {}

    for prop in properties:
        if prop not in df.columns:
            continue

        out_lr_rows = []
        out_rf_rows = []

        # unique non-null values
        values = df[prop].dropna().unique().tolist()
        # Optional: sort values nicely
        try:
            values = sorted(values, key=lambda x: str(x))
        except Exception:
            pass

        for val in values:
            mask = (df[prop] == val)
            subset = df.loc[mask].copy()
            n = len(subset)
            if n < min_n:
                continue

            # target and groups for subset
            y = (subset[label_col].astype(str).str.upper() == "ON").astype(int).values
            if y.min() == y.max():
                # only one class → skip
                continue

            X = subset[feats].copy()
            groups = subset[group_col].values if group_col in subset.columns else np.zeros(n, dtype=int)

            gkf = GroupKFold(n_splits=min(n_splits, len(np.unique(groups))))
            lr_imp_fold = []
            rf_imp_fold = []

            for tr_idx, te_idx in gkf.split(X, y, groups):
                Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
                ytr, yte = y[tr_idx], y[te_idx]

                # ---- LR fit + standardized ORs (coef on scaled features) ----
                lr_pipe.fit(Xtr, ytr)
                lr_clf = lr_pipe.named_steps["clf"]
                # Coefs correspond to scaled features (thanks to StandardScaler in the pipeline)
                coef = lr_clf.coef_.ravel()
                or_per_sd = np.exp(coef)  # OR for +1 SD
                lr_imp_fold.append(or_per_sd)

                # ---- RF fit + permutation importance on validation split ----
                rf_pipe.fit(Xtr, ytr)
                rf_model = rf_pipe.named_steps["clf"]
                # permutation importance on validation set
                # use the whole pipeline to transform imputation, but permute only features
                Xte_imp = rf_pipe.named_steps["imputer"].transform(Xte)
                perm = permutation_importance(
                    rf_model, Xte_imp, yte,
                    n_repeats=10, random_state=rng, n_jobs=-1
                )
                rf_imp_fold.append(perm.importances_mean)

            # Aggregate over folds
            if lr_imp_fold:
                lr_imp = np.vstack(lr_imp_fold)
                lr_mean = lr_imp.mean(axis=0)
                lr_std  = lr_imp.std(axis=0, ddof=1) if lr_imp.shape[0] > 1 else np.zeros_like(lr_mean)
                for f, m, s in zip(feats, lr_mean, lr_std):
                    out_lr_rows.append({"value": val, "n": n, "feat": f, "mean": m, "std": s})

            if rf_imp_fold:
                rf_imp = np.vstack(rf_imp_fold)
                rf_mean = rf_imp.mean(axis=0)
                rf_std  = rf_imp.std(axis=0, ddof=1) if rf_imp.shape[0] > 1 else np.zeros_like(rf_mean)
                for f, m, s in zip(feats, rf_mean, rf_std):
                    out_rf_rows.append({"value": val, "n": n, "feat": f, "mean": m, "std": s})

        # Build tidy DataFrames
        res_prop = {}
        if out_lr_rows:
            df_lr = pd.DataFrame(out_lr_rows)
            # sort each value block by importance descending
            df_lr = df_lr.sort_values(["value","mean"], ascending=[True, False]).reset_index(drop=True)
            res_prop["LR"] = df_lr
        if out_rf_rows:
            df_rf = pd.DataFrame(out_rf_rows)
            df_rf = df_rf.sort_values(["value","mean"], ascending=[True, False]).reset_index(drop=True)
            res_prop["RF"] = df_rf

        if res_prop:
            results[prop] = res_prop

    return results

# -----------------------------
# Pretty printer (optional)
# -----------------------------
def print_importance_summary(imp_dict: dict, top_k: int = 10, dec: int = 4):
    for prop, models in imp_dict.items():
        print(f"\n=== Feature importance by '{prop}' ===")
        for model_name, df_imp in models.items():
            print(f"\n-- {model_name} (top {top_k} per value) --")
            for val in df_imp["value"].unique():
                block = df_imp[df_imp["value"] == val].head(top_k)
                n = int(block["n"].iloc[0])
                print(f"\nValue: {val} (n={n})")
                disp = block[["feat","mean","std"]].copy()
                print(disp.round({"mean":dec, "std":dec}).to_string(index=False))


# task 2

In [8]:
features_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs2/windows_with_animation_details2.csv")
# 1) Choose your features list (your FEATS from Task 2)
FEATS = [
    'fix_rate_per_s','fix_median_dur',
    'gaze_dispersion_total',
    'saccade_mean_mag','saccade_rate_per_s',
    'dir_change_rate','axis_ratio',
    'x_mean','x_dir_change_rate','left_ratio','right_ratio','x_entropy'
]

# 2) Run importance aggregation by animation properties
imp = feature_importance_by_property(
    features_df,                 # your per-window table
    feats=FEATS,
    label_col="label",
    properties=["ease","direction","delay"],   # pick any subset you have
    group_col="User ID (file)",  # grouping for CV
    n_splits=5,
    min_n=120,                   # skip tiny subgroups
    random_state=42
)

# 3) Print a quick summary
print_importance_summary(imp, top_k=8)


=== Feature importance by 'ease' ===

-- LR (top 8 per value) --

Value: Ease-in (n=848)
                 feat   mean    std
          right_ratio 1.7305 0.5704
   saccade_rate_per_s 1.2572 0.3867
       fix_median_dur 1.2403 0.2460
            x_entropy 1.2167 0.3578
    x_dir_change_rate 1.1110 0.1225
       fix_rate_per_s 1.1015 0.1859
      dir_change_rate 1.0430 0.0339
gaze_dispersion_total 0.9659 0.0008

Value: Ease-out (n=847)
             feat   mean    std
  dir_change_rate 1.5853 0.4490
   fix_rate_per_s 1.5786 0.4074
      right_ratio 1.3786 0.3173
 saccade_mean_mag 1.3201 0.1352
   fix_median_dur 1.1975 0.1720
x_dir_change_rate 0.9902 0.1972
        x_entropy 0.9749 0.0047
       axis_ratio 0.9372 0.0405

-- RF (top 8 per value) --

Value: Ease-in (n=848)
                 feat   mean    std
               x_mean 0.1239 0.0615
           axis_ratio 0.0425 0.0472
gaze_dispersion_total 0.0314 0.0422
       fix_rate_per_s 0.0291 0.0312
       fix_median_dur 0.0273 0.0658
    x

# task 3

In [9]:
features_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs3/windows_with_animation_details3.csv")
# 1) Choose your features list (your FEATS from Task 2)
FEATS = [
    # base (your strong ones)
    "fix_rate_per_s",
    "fix_median_dur",
    "fix_efficiency",
    "gaze_dispersion_total",
    "pupil_mad",
    "pupil_slope",
    "pupil_var",
    "low_vel_frac",
    "fix_stability",
    "pupil_roll_std_mean",
]

# 2) Run importance aggregation by animation properties
imp = feature_importance_by_property(
    features_df,                 # your per-window table
    feats=FEATS,
    label_col="label",
    properties=["ease","direction","delay"],   # pick any subset you have
    group_col="User ID (file)",  # grouping for CV
    n_splits=5,
    min_n=120,                   # skip tiny subgroups
    random_state=42
)

# 3) Print a quick summary
print_importance_summary(imp, top_k=8)


=== Feature importance by 'ease' ===

-- LR (top 8 per value) --

Value: Ease-in (n=745)
                 feat   mean    std
       fix_rate_per_s 2.8966 0.5166
  pupil_roll_std_mean 1.3068 0.5886
         low_vel_frac 1.2721 0.0110
       fix_median_dur 1.2165 0.0858
          pupil_slope 1.0741 0.0039
            pupil_mad 1.0579 0.0914
        fix_stability 1.0062 0.0985
gaze_dispersion_total 0.9902 0.0775

Value: Ease-out (n=651)
               feat   mean    std
     fix_rate_per_s 1.6930 0.0253
       low_vel_frac 1.2582 0.0672
     fix_efficiency 1.2150 0.2798
     fix_median_dur 1.1345 0.1099
      fix_stability 0.9429 0.0628
pupil_roll_std_mean 0.9292 0.0737
          pupil_mad 0.9102 0.3392
          pupil_var 0.8425 0.0396

-- RF (top 8 per value) --

Value: Ease-in (n=745)
                 feat    mean    std
       fix_rate_per_s  0.0331 0.0449
       fix_median_dur  0.0298 0.0614
         low_vel_frac  0.0106 0.0382
gaze_dispersion_total -0.0048 0.0270
  pupil_roll_std_m

In [ ]:
features_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs1/windows_with_animation_details1.csv")
# 1) Choose your features list (your FEATS from Task 2)
FEATS = [
    'fix_rate_per_s','fix_median_dur',
    'gaze_dispersion_total',
    'saccade_mean_mag','saccade_rate_per_s',
    'dir_change_rate','axis_ratio',
    'x_mean','x_dir_change_rate','left_ratio','right_ratio','x_entropy'
]

# 2) Run importance aggregation by animation properties
imp = feature_importance_by_property(
    features_df,                 # your per-window table
    feats=FEATS,
    label_col="label",
    properties=["ease","direction","delay"],   # pick any subset you have
    group_col="User ID (file)",  # grouping for CV
    n_splits=5,
    min_n=120,                   # skip tiny subgroups
    random_state=42
)

# 3) Print a quick summary
print_importance_summary(imp, top_k=8)